In [ ]:
"""Novel-family Decider inference on 2×T4; translation and model private Kaggle inputs."""
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

ROOT = Path("/tmp/innovation-group7")
OUT = Path("/tmp/humanoid-production")
SAVED = Path("/kaggle/working")
SAVED.mkdir(parents=True, exist_ok=True)
PINNED_COMMIT = "REPLACE_GIT_SHA"

def check(cmd, **kw):
    print("RUNNING:", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, **kw)

check(["git", "clone", "--no-checkout", "https://github.com/roccoangelella/Innovation-management-group-7.git", str(ROOT)])
check(["git", "-C", str(ROOT), "checkout", "--detach", PINNED_COMMIT])
# The translated novel family corpus is an account-private Kaggle data mount.
import hashlib
input_root = Path("/kaggle/input")
sources = sorted(input_root.rglob("novel_humanoid_patents_en.csv"))
summaries = sorted(input_root.rglob("novel_translation_summary.json"))
if len(sources) != 1 or len(summaries) != 1 or sources[0].parent != summaries[0].parent:
    raise RuntimeError("Expected one matching novel English corpus and summary in Kaggle inputs")
source, summary = sources[0], summaries[0]
translated = json.loads(summary.read_text(encoding="utf-8"))
count = translated.get("completed")
if (type(count) is not int or count <= 1 or translated.get("incomplete") != 0
        or translated.get("eligible") != count or translated.get("stop_reason") is not None):
    raise RuntimeError("Novel translation must be complete before inference")
with source.open("rb") as stream:
    csv_sha = hashlib.file_digest(stream, "sha256").hexdigest()
if csv_sha != translated["final_csv_sha256"]:
    raise RuntimeError("Novel translated CSV SHA256 does not match translation summary")
print("NOVEL_CORPUS_READY", count, csv_sha, translated["source_sha256"], flush=True)
print("KAGGLE_INPUT_DIRS", sorted(str(p) for p in input_root.iterdir()), flush=True)
checkpoint = Path("/tmp/pplx-private-model")
manifest_paths = list(input_root.rglob("snapshot_manifest.json")) if input_root.is_dir() else []
print("FOUND_PRIVATE_MODEL_MANIFESTS", len(manifest_paths), flush=True)
if len(manifest_paths) == 1:
    manifest_path = manifest_paths[0]
    snapshot_files = json.loads(manifest_path.read_text(encoding="utf-8"))
    for entry in snapshot_files:
        destination = (checkpoint / entry["path"]).resolve()
        if not destination.is_relative_to(checkpoint.resolve()):
            raise ValueError("Unsafe checkpoint filename")
        source_file = manifest_path.parent / entry["flat_name"]
        if not source_file.is_file():
            raise FileNotFoundError(source_file)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_file, destination)
else:
    raise FileNotFoundError("Expected exactly one mounted private model snapshot_manifest.json; inspect KAGGLE_INPUT_DIRS")
assert (checkpoint / "source" / "src").exists(), "Model source not found"
print("MODEL_STAGED", flush=True)

# Resume only from an explicitly mounted, dataset-specific checkpoint.
# Never touch the original 237538-family production journal.
resume_enabled = "REPLACE_RESUME_ENABLED" == "true"
resume_roots = sorted(p.parent for p in input_root.rglob("production_manifest.json")
                      if "humanoid-novel-resume-" in p.parent.name)
if resume_enabled:
    if len(resume_roots) != 1:
        raise RuntimeError("Resume requested but exactly one novel journal mount was not found")
    resume_root = resume_roots[0]
    expected = {"production_manifest.json", "session_summary.json"}
    expected |= {f"worker_{i}/{name}" for i in (0, 1)
                 for name in ("predictions.jsonl", "run_metadata.json")}
    actual = {p.relative_to(resume_root).as_posix() for p in resume_root.rglob("*") if p.is_file()}
    if actual != expected:
        raise RuntimeError(f"Novel resume journal layout mismatch: {actual ^ expected}")
    OUT.mkdir(parents=True, exist_ok=True)
    for rel in sorted(expected):
        saved_file = resume_root / rel
        if not saved_file.is_file() or saved_file.stat().st_size == 0:
            raise RuntimeError(f"Empty or missing novel checkpoint file: {rel}")
        target = OUT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(saved_file, target)
    print("NOVEL_CHECKPOINT_RESTORED", flush=True)
elif resume_roots:
    raise RuntimeError("Novel checkpoint mounted but resume flag is off: refusing ambiguous start")

# Direct W&B logging: prefer Kaggle Secrets, otherwise an owner-private
# single-run credential dataset created by GitHub Actions (not in notebook code).
auth_paths = sorted(input_root.rglob("wandb_auth_token.txt"))
if len(auth_paths) == 1:
    # Prefer the private single-run dataset provisioned by GitHub Actions.
    secret = auth_paths[0].read_text(encoding="utf-8").strip()
elif len(auth_paths) > 1:
    raise RuntimeError("Ambiguous private W&B credential mounts; refusing to start")
else:
    # Kaggle's secrets backend may raise Kaggle-specific BackendError.
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        secret = None
    if not secret:
        raise RuntimeError("No Kaggle W&B secret or private auth mount; refusing unmonitored inference")
if len(secret) < 20:
    raise ValueError("Invalid W&B credential")

# Kaggle's Python 3.13 image does not support python -m venv/ensurepip here.
# Install W&B into an isolated import directory without creating a venv.
# The inference workers do NOT inherit this PYTHONPATH.
monitor_site = Path("/tmp/wandb-monitor-packages")
check([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
       "--no-input", "--upgrade", "--target", str(monitor_site), "wandb==0.31.0"])
monitor_ready = Path("/tmp/wandb_monitor_ready.json")
monitor_log = SAVED / "wandb_monitor.log"
monitor_env = os.environ.copy()
monitor_env.update({
    "WANDB_API_KEY": secret,
    "WANDB_BASE_URL": "https://api.forge.coreweave.com",
    "WANDB_MODE": "online",
    "PYTHONPATH": str(monitor_site) + os.pathsep + monitor_env.get("PYTHONPATH", ""),
})
secret = None
monitor_stream = monitor_log.open("w", encoding="utf-8")
monitor = subprocess.Popen([
    sys.executable, "-u", str(ROOT / "kaggle/wandb_live.py"),
    "--production-dir", str(OUT),
    "--ready-file", str(monitor_ready),
    "--run-name", "novel-dual-t4-direct-REPLACE_RUN_ID",
    "--interval", "30",
    "--total", str(count),
], cwd=ROOT, env=monitor_env, stdout=monitor_stream, stderr=subprocess.STDOUT)
monitor_env.pop("WANDB_API_KEY", None)
del monitor_env
import time
for _ in range(120):
    if monitor_ready.is_file():
        break
    if monitor.poll() is not None:
        raise RuntimeError("W&B direct monitor exited before opening a run; see wandb_monitor.log in Kaggle outputs")
    time.sleep(1)
else:
    monitor.terminate()
    raise RuntimeError("W&B direct monitor failed to initialize within 120 seconds")
print("WANDB_DIRECT_URL", json.loads(monitor_ready.read_text())["url"], flush=True)

try:
    check([
        sys.executable, "-u", str(ROOT / "kaggle/production_run.py"),
        "--input", str(source),
        "--translation-summary", str(summary),
        "--production-dir", str(OUT),
        "--checkpoint", str(checkpoint),
        "--skip-download",
        "--session-minutes", "300",
    ], cwd=ROOT)
finally:
    monitor.terminate()
    try:
        monitor.wait(timeout=60)
    except subprocess.TimeoutExpired:
        monitor.kill()
        monitor.wait()
    monitor_stream.close()
    print("WANDB_DIRECT_MONITOR_EXIT", monitor.returncode, flush=True)
    # Always save the audit/progress state, even on a failed model run.
    with zipfile.ZipFile(SAVED / "humanoid-checkpoint.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
        for name in [
            "production_manifest.json", "worker_0/run_metadata.json",
            "worker_1/run_metadata.json", "worker_0/predictions.jsonl",
            "worker_1/predictions.jsonl", "session_summary.json",
            "production_summary.json",
        ]:
            f = OUT / name
            if f.is_file():
                archive.write(f, arcname=name)
    for name in ("predictions.csv", "filtered_patents.zip", "production_summary.json", "session_summary.json"):
        f = OUT / name
        if f.is_file():
            shutil.copy2(f, SAVED / name)
    print("CHECKPOINT_SAVED", (SAVED / "humanoid-checkpoint.zip").stat().st_size, flush=True)

# Kaggle may finish a time-limited session successfully without complete inference.
# The saved journal remains available through the GitHub Actions artifact.
final_summary = OUT / "production_summary.json"
if not final_summary.is_file():
    raise RuntimeError("Novel inference incomplete: checkpoint saved; resume from this Actions run")
result = json.loads(final_summary.read_text(encoding="utf-8"))
print("NOVEL_PRODUCTION_SUMMARY", json.dumps(result, sort_keys=True), flush=True)
